In [ ]:
#!/usr/bin/env python3
"""
Threat Analyst for Michael AI using CrewAI and a local Ollama model (llama3.2).

This script inspects vehicle and perimeter telemetry for anomalies, then uses a local
Ollama model to reason about the incidents and recommend actions.

Requirements:
  pip install crewai requests
  ollama serve
  ollama pull llama3.2

Run:
  python Untitled1.ipynb --telemetry telemetry.json

"""

import argparse
import json
import os
import sys
from typing import Any, Dict, List, Optional

import requests

# CrewAI is optional here: the script includes a lightweight fallback if the SDK is not installed.
try:
    from crewai import Agent, Crew, Task
except Exception:  # pragma: no cover
    Agent = None
    Crew = None
    Task = None

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "llama3.2"


class OllamaLLM:
    def __init__(self, base_url: str = OLLAMA_URL, model: str = MODEL_NAME):
        self.base_url = base_url.rstrip("/")
        self.model = model

    def generate(self, prompt: str, temperature: float = 0.2, max_tokens: int = 800) -> str:
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "temperature": temperature,
            "options": {"num_predict": max_tokens},
        }
        response = requests.post(f"{self.base_url}/api/generate", json=payload, timeout=60)
        response.raise_for_status()
        data = response.json()
        return data.get("response", str(data))


class ThreatAnalystAgent:
    def __init__(self, llm: OllamaLLM, name: str = "Michael AI Threat Analyst"):
        self.name = name
        self.llm = llm

    def inspect_telemetry(self, telemetry: Dict[str, Any]) -> Dict[str, Any]:
        anomalies = []

        # Vehicle telemetry checks
        for vehicle in telemetry.get("vehicles", []):
            vid = vehicle.get("id", "unknown")
            speed = vehicle.get("speed_kph")
            if speed is not None and speed > vehicle.get("speed_limit_kph", 120) * 1.2:
                anomalies.append({
                    "type": "overspeed",
                    "severity": "high",
                    "vehicle_id": vid,
                    "details": f"Vehicle {vid} exceeded speed limit by >20%"
                })

            if vehicle.get("location") and vehicle.get("location") == telemetry.get("last_perimeter_event", {}).get("location"):
                anomalies.append({
                    "type": "vehicle_near_perimeter",
                    "severity": "medium",
                    "vehicle_id": vid,
                    "details": f"Vehicle {vid} is close to a reported perimeter event"
                })

        # Perimeter telemetry checks
        for sensor in telemetry.get("perimeter_sensors", []):
            if sensor.get("status") in {"tampered", "alarm", "breach"}:
                anomalies.append({
                    "type": "perimeter_alert",
                    "severity": "critical" if sensor.get("status") == "breach" else "high",
                    "sensor_id": sensor.get("id", "unknown"),
                    "details": f"Perimeter sensor {sensor.get('id','unknown')} status={sensor.get('status')}"
                })

        # Simple correlation logic
        if anomalies:
            prompt = f"""
You are the Threat Analyst for Michael AI. Examine the telemetry and anomalies below.
Identify likely threats, prioritize them, and suggest quick actions.

Telemetry:
{json.dumps(telemetry, indent=2, default=str)}

Anomalies:
{json.dumps(anomalies, indent=2, default=str)}

Return valid JSON with keys: 
- summary
- threat_level
- prioritized_findings
- recommended_actions
- notes
"""
            model_response = self.llm.generate(prompt, temperature=0.1, max_tokens=1200)
            try:
                parsed = json.loads(model_response)
            except json.JSONDecodeError:
                parsed = {
                    "summary": "Model output not valid JSON; manual review required.",
                    "threat_level": "unknown",
                    "prioritized_findings": [model_response],
                    "recommended_actions": ["Review telemetry manually"],
                    "notes": ["The LLM returned non-JSON output; parse failed."]
                }
        else:
            parsed = {
                "summary": "No definite anomalies detected. Continue routine monitoring.",
                "threat_level": "low",
                "prioritized_findings": [],
                "recommended_actions": ["Continue standard monitoring."],
                "notes": ["No threshold violations observed."]
            }

        return {
            "agent": self.name,
            "anomalies": anomalies,
            "analysis": parsed,
        }


class MichaelAICrew:
    def __init__(self, llm: OllamaLLM):
        self.llm = llm
        self.agent = ThreatAnalystAgent(llm=llm)

    def run(self, telemetry: Dict[str, Any]) -> Dict[str, Any]:
        return self.agent.inspect_telemetry(telemetry)


def load_telemetry(path: Optional[str]) -> Dict[str, Any]:
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    if not path:
        raw = sys.stdin.read().strip()
        if not raw:
            raise ValueError("No telemetry JSON provided. Use --telemetry or stdin.")
        return json.loads(raw)
    raise FileNotFoundError(f"Telemetry file not found: {path}")


def main() -> None:
    parser = argparse.ArgumentParser(description="Threat Analyst agent for Michael AI with a local Ollama model")
    parser.add_argument("--telemetry", type=str, default=None, help="Path to JSON telemetry file")
    parser.add_argument("--model-url", type=str, default=OLLAMA_URL, help="Ollama base URL, default http://localhost:11434")
    parser.add_argument("--model", type=str, default=MODEL_NAME, help="Ollama model name, default llama3.2")
    args = parser.parse_args()

    llm = OllamaLLM(base_url=args.model_url, model=args.model)
    crew = MichaelAICrew(llm=llm)
    telemetry = load_telemetry(args.telemetry)
    result = crew.run(telemetry)
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()
